# 15 · Linear channel equalization

A channel that is not flat in frequency (multipath, bandwidth limits,
residual dispersion) spreads each symbol into its neighbours: inter-symbol
interference.  If the channel response is known, a **linear equalizer**
undoes it in the frequency domain.  This notebook covers the fixed
(non-adaptive) case:

1. the ISI channel model;
2. **zero forcing** (ZF): invert the channel exactly;
3. **MMSE**: invert it as far as the noise allows;
4. measuring the channel: data-aided **channel sounding**, and equalizing
   with the measured response.

Adaptive equalizers, which learn the inverse from the data and track
changes, follow in [16](16-adaptive-equalization.ipynb).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import commkit as ck
from commkit import Constellation
from commkit.backend import to_device

DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
print(DEVICE)

ck.plotting.apply_default_theme()
C16 = Constellation.qam(16)


def host(a):
    return to_device(a, "cpu")


tx = ck.generate(C16, num_symbols=2**15, symbol_rate=32e9, rng=1, device=DEVICE)

## 1. The ISI channel

At one sample per symbol, a linear channel is a causal FIR filter,

$$r[n] = \sum_k h[k]\, s[n-k] + w[n],$$

with frequency response `H(f) = Σ h[k] e^{-j2πfk}`.  Three channels of
increasing difficulty: a mild one, and two with a spectral notch of 20 and
30 dB, the kind of fade that two paths of nearly equal strength create.

In [ ]:
channels = {
    "mild": np.array([1, 0.3 + 0.2j, -0.1], dtype=np.complex64),
    "20 dB notch": np.array([1, 0.9j], dtype=np.complex64),
    "30 dB notch": np.array([1, 0.97j], dtype=np.complex64),
}
ESN0 = 18.0


def through(h):
    xp = tx.xp
    r = xp.convolve(tx.samples, xp.asarray(h))[: tx.samples.shape[-1]]
    # The noise is set against the transmitted power (1), not the received one.
    return ck.impairments.apply_awgn(
        tx.replace(samples=r), esn0_db=ESN0, signal_power=1.0, rng=2
    )


fig, (ax_f, ax_c) = plt.subplots(1, 2, figsize=(12, 4.5))
f = np.fft.fftshift(np.fft.fftfreq(1024))
for name, h in channels.items():
    ax_f.plot(
        f, 20 * np.log10(np.abs(np.fft.fftshift(np.fft.fft(h, 1024)))), label=name
    )
ax_f.set_xlabel("Frequency [symbol rates]")
ax_f.set_ylabel("|H(f)| [dB]")
ax_f.legend()
ck.plotting.plot_constellation(
    through(channels["mild"]), ax=ax_c, title="Through the mild channel"
)
plt.show()

## 2. Zero forcing

The zero-forcing equalizer applies `W(f) = 1 / H(f)`: the cascade is flat
and the ISI is gone.  `zf_equalizer(r, h)` does this blockwise in the
frequency domain (overlap-save), for SISO `(L,)` or MIMO `(C, C, L)`
impulse responses.  Its weakness is the noise: where `|H|` is small, `1/|H|`
is large, and the noise in that band is amplified with it.

In [ ]:
print(f"Es/N0 = {ESN0:.0f} dB")
for name, h in channels.items():
    r = through(h)
    zf = ck.equalization.zf_equalizer(r, h)
    print(
        f"{name:12s}: before {ck.metrics.snr(r):5.2f} dB, after ZF {ck.metrics.snr(zf):5.2f} dB"
    )

## 3. MMSE

The minimum-mean-square-error equalizer balances ISI against noise:

$$W(f) = \frac{H^*(f)}{|H(f)|^2 + \sigma^2},$$

which is `1/H` where the channel is strong and stays bounded in a notch,
accepting a little residual ISI there rather than amplifying the noise.
`zf_equalizer(..., noise_variance=σ²)` switches to it; `σ²` is the noise
variance relative to the signal power, `10^(-Es/N0 / 10)`.

In [ ]:
sigma2 = 10 ** (-ESN0 / 10)
for name, h in channels.items():
    r = through(h)
    zf = ck.metrics.snr(ck.equalization.zf_equalizer(r, h))
    mmse = ck.metrics.snr(ck.equalization.zf_equalizer(r, h, noise_variance=sigma2))
    print(f"{name:12s}: ZF {zf:5.2f} dB, MMSE {mmse:5.2f} dB")
ck.plotting.plot_zf_equalizer_response(
    channels["30 dB notch"],
    noise_variance=sigma2,
    sampling_rate=tx.symbol_rate,
    title="30 dB notch: ZF against MMSE",
    show=True,
)

## 4. Measuring the channel

The equalizers above were handed the true `h`.  A receiver measures it by
**channel sounding**: send known data `x`, receive `y`, and estimate
`H(f) = S_yx(f) / S_xx(f)` from averaged cross- and auto-spectra (the
Welch H1 estimator).  `estimate_transfer_function(x, y)` returns `H(f)` in
FFT-bin order (with `num_taps=`, a truncated impulse response centred on
lag 0); for two polarizations it returns the full 2x2 response in one shot.

In [ ]:
n_fft = 256
fig, ax = plt.subplots(figsize=(8, 4))
for name, h in channels.items():
    est = host(ck.equalization.estimate_transfer_function(tx, through(h), n_fft=n_fft))
    true = np.fft.fft(h, n_fft)
    order = np.argsort(np.fft.fftfreq(n_fft))
    line = ax.plot(
        np.fft.fftfreq(n_fft)[order],
        20 * np.log10(np.abs(est[order])),
        label=f"{name}, estimated",
    )
    ax.plot(
        np.fft.fftfreq(n_fft)[order],
        20 * np.log10(np.abs(true[order])),
        "--",
        color=line[0].get_color(),
    )
    print(f"{name:12s}: max |H_est - H| = {np.max(np.abs(est - true)):.3f}")
ax.set_xlabel("Frequency [symbol rates]")
ax.set_ylabel("|H(f)| [dB]")
ax.set_title("Channel sounding (dashed: true response)")
ax.legend()
plt.show()

### Sound, then equalize

With `num_taps=L` the estimate is an impulse response with **lag 0 in the
middle**, at index `L // 2`: the taps before it are precursors, which a
measured channel can have (a sampling phase off the peak, an anticausal
filter).  `zf_equalizer` reads index 0 as lag 0 unless told otherwise, so
pass `center=L // 2`.  Without it the output comes out `L // 2` symbols early.

In [ ]:
L = 31
for name, h in channels.items():
    r = through(h)
    h_est = ck.equalization.estimate_transfer_function(tx, r, num_taps=L)
    known = ck.equalization.zf_equalizer(r, h, noise_variance=sigma2)
    sounded = ck.equalization.zf_equalizer(
        r, h_est, noise_variance=sigma2, center=L // 2
    )
    print(
        f"{name:12s}: MMSE with true h {ck.metrics.snr(known):5.2f} dB, "
        f"with sounded h {ck.metrics.snr(sounded):5.2f} dB"
    )

**Why adaptive equalizers?**  A fixed equalizer needs the channel first
and assumes it stays put.  Real optical channels change: the polarization
rotates, PMD drifts, and the residual dispersion is unknown.  An adaptive
equalizer learns `W` from the data directly and keeps following it.

**Summary**

* ZF inverts `H(f)`; it removes all ISI but amplifies the noise in notches.
* MMSE adds `σ²` to the denominator and is never worse than ZF.
* `estimate_transfer_function` sounds the channel from known data; its
  `num_taps=L` response goes into `zf_equalizer(..., center=L // 2)`.

**Next:** [16 · Adaptive equalization](16-adaptive-equalization.ipynb).